# 04 · Team profiles

How do teams differ in the way they build drives, are those differences real, and how much do they matter?

1. **Style DNA** per team-season, adjusted for game script (a team that trails a lot hurries up, whatever its taste).
2. **Is it a real team trait?** Reliability and year-to-year persistence (~143 drives per team-season is small).
3. **Similarity** between team-seasons.
4. **Change:** does a new head coach or a new primary QB move a team's style?
5. **Execution vs. style:** how much of the gap between offenses is *how they call drives* vs. *how well they run them*?

*Limits:* head coaches and primary QBs are in the data; offensive coordinators are not, so a new play-caller under the same head coach is invisible.

In [ ]:
import numpy as np, pandas as pd
import plotly.express as px, plotly.graph_objects as go
from drivedna.models.cluster import REPORTS_DIR, STYLE_ORDER
from drivedna.analysis.openings import OPENING_ORDER

dna = pd.read_csv(REPORTS_DIR / "team_season_dna.csv")
rel = pd.read_csv(REPORTS_DIR / "dna_reliability.csv")
nbrs = pd.read_csv(REPORTS_DIR / "team_neighbors_latest.csv")
events = pd.read_csv(REPORTS_DIR / "change_events.csv")
summ = pd.read_csv(REPORTS_DIR / "change_event_summary.csv")
dec = pd.read_csv(REPORTS_DIR / "performance_decomposition.csv")
pers = pd.read_csv(REPORTS_DIR / "component_persistence.csv")
dec["label"] = dec["team"] + " " + dec["season"].astype(str)
dna["label"] = dna["team"] + " " + dna["season"].astype(str)
print(f"{len(dna)} team-seasons; {dna['n_drives'].mean():.0f} drives each on average (min {dna['n_drives'].min()}, max {dna['n_drives'].max()})")

## 1 · Style DNA, adjusted for game script
For each style: the team's actual share of drives minus the share expected from the situations its drives started in
(field position, score, time left, quarter, opponent). Positive = the team uses that style more than its situations predict.

In [ ]:
latest = dna[dna["season"] == dna["season"].max()].set_index("team")
cols = [f"drive_{s}_excess" for s in STYLE_ORDER]
fig = px.imshow(latest[cols].rename(columns=lambda c: c.replace("drive_", "").replace("_excess", "")).sort_index() * 100,
                color_continuous_scale="RdBu_r", zmin=-25, zmax=25, aspect="auto", text_auto=".0f",
                title=f"{int(dna['season'].max())}: share of drives in each style, percentage points above/below what the situations predict")
fig.update_layout(height=900); fig.show()

In [ ]:
long = dna.melt(id_vars=["label", "n_drives"], value_vars=[c for c in dna.columns if c.endswith("_excess")], var_name="style", value_name="excess")
long["style"] = long["style"].str.replace("_excess", "")
print("Most extreme team-season tendencies (percentage points above expected):")
print(long.reindex(long["excess"].abs().sort_values(ascending=False).index).head(14).assign(excess=lambda d: (d.excess * 100).round(0))[["label", "style", "excess"]].to_string(index=False))

## 2 · Is it a real team trait?
**Split-half reliability:** randomly split a team-season's games in two; how well do the halves agree across team-seasons (Spearman-Brown corrected)?
**Year-to-year:** correlation of the same team's tendency in consecutive seasons. If year-to-year is well below reliability, the trait changes (coaches, QBs, personnel).

In [ ]:
r = rel.copy(); r["style"] = r["kind"] + ": " + r["style"]
print(r[["style", "reliability", "year_to_year_corr"]].round(2).to_string(index=False))
m = r.melt(id_vars="style", value_vars=["reliability", "year_to_year_corr"])
px.bar(m, x="value", y="style", color="variable", barmode="group", height=520,
       title="Team style tendencies are real but only partly persistent (the Deep-shot opener tendency is mostly noise)").show()

## 3 · Similarity
Cosine similarity of the script-adjusted tendencies (robust-scaled so extreme no-huddle teams don't dominate). Nearest other-team neighbors for each team's latest season:

In [ ]:
top = nbrs[nbrs["rank"] <= 3].groupby("team_season").apply(lambda g: ", ".join(f"{a} ({c:.2f})" for a, c in zip(g["similar_to"], g["cosine"])))
print(top.to_string())

In [ ]:
from sklearn.decomposition import PCA
excess = [c for c in dna.columns if c.endswith("_excess")]
z = (dna[excess] - dna[excess].median()) / ((dna[excess] - dna[excess].median()).abs().median() * 1.4826)
pc = PCA(2).fit_transform(z.clip(-5, 5))
mp = dna.assign(pc1=pc[:, 0], pc2=pc[:, 1])
px.scatter(mp, x="pc1", y="pc2", color="season", hover_name="label", height=650,
           title="Team-seasons in style space (first two principal components of script-adjusted tendencies)").show()

## 4 · Do coaching and QB changes move a team's style?
Distance (in robust-z units) between a team's style tendencies and the previous season's, split by what changed. The baseline is large because
a single season's DNA is noisy, so the question is how far above 'no change' each event sits.

In [ ]:
print(summ.round(2).to_string(index=False))
fig = go.Figure(go.Scatter(x=summ["mean_shift"], y=summ["event"], mode="markers", marker=dict(size=11),
    error_x=dict(type="data", symmetric=False, array=summ["hi"] - summ["mean_shift"], arrayminus=summ["mean_shift"] - summ["lo"])))
fig.update_layout(title="Season-to-season style shift by event (95% bootstrap CI)", height=330, yaxis=dict(autorange="reversed")); fig.show()

In [ ]:
cols_show = ["team", "season", "prev_coach", "coach", "prev_qb", "qb", "event", "distance", "biggest_shift", "shift_direction"]
print("Largest single-season style shifts:"); print(events.sort_values("distance", ascending=False).head(10)[cols_show].round(1).to_string(index=False))

In [ ]:
# one coach's fingerprint across teams: no-huddle tendency (percentage points above expected)
tr = dna[dna["team"].isin(["ARI", "WAS"])][["team", "season", "coach", "drive_No-huddle_excess"]].copy()
tr["no_huddle_excess_pp"] = (tr["drive_No-huddle_excess"] * 100).round(0)
print(tr.pivot(index="season", columns="team", values="no_huddle_excess_pp").to_string())
print(); print(dna[dna["team"].isin(["ARI", "WAS"])].pivot(index="season", columns="team", values="coach").to_string())

## 5 · Execution vs. style
Each team's points per drive above the league, split into: **situation** (field position, score, time, opponent), **style** (what the opening styles it chose
are worth in those situations: Phase 4's association, so observational) and **execution** (actual minus predicted given all that: QB, talent,
scheme quality and luck). The model for this deliberately does *not* control for the team's own offensive strength, which would absorb execution.

In [ ]:
print(pers.round(3).to_string(index=False))
print("\nShare of the variance in team points/drive vs. league explained by each component:",
      {c: round(dec[c].var() / dec["total_vs_league"].var(), 2) for c in ["situation_part", "style_part", "exec_part"]})
print("Std across team-seasons (points per drive):", dec[["situation_part", "style_part", "exec_part"]].std().round(3).to_dict())
print(f"Typical standard error of one team-season's execution estimate: {dec['exec_se'].mean():.2f} points/drive")

In [ ]:
cols = ["label", "points_per_drive", "situation_part", "style_part", "exec_part", "exec_se"]
print("Best execution:"); print(dec.sort_values("exec_part", ascending=False).head(8)[cols].round(2).to_string(index=False))
print("\nWorst execution:"); print(dec.sort_values("exec_part").head(8)[cols].round(2).to_string(index=False))

In [ ]:
px.scatter(dec, x="style_part", y="exec_part", hover_name="label", color="season", height=560,
           labels={"style_part": "style component (points/drive)", "exec_part": "execution (points/drive)"},
           title="Teams differ far more in execution than in the value of their style choices").show()

## Takeaways

**1. Team style tendencies are real, but only partly persistent.** Split-half reliability is 0.5-0.9 for most tendencies and year-to-year correlation is 0.26-0.55, so roughly 60% of a team's true tendency carries into the next season. **No-huddle is the most team-specific trait** (reliability 0.93, year-to-year 0.55). The **Deep-shot opening tendency is mostly noise at the team-season level** (reliability 0.30, year-to-year 0.14), so the best opener from Phase 4 is not something teams reliably "have."

**2. The DNA follows play-callers.** Arizona's no-huddle tendency (percentage points above what its situations predict) runs +28, +48, +44, +33 in 2019-22 under Kliff Kingsbury, drops to -4 in 2023 when he leaves, and **reappears in Washington at +73 and +66 in 2024-25**, where the head coach in the data is Dan Quinn. Kingsbury was Washington's offensive coordinator from 2024, a role this data cannot see; the fingerprint found him anyway. Other extreme seasons are the 2016 Giants (+55) and 49ers (+53) and Detroit 2017 (+31). Nearest-neighbor similarity recovers the same family (WAS 2025 ~ SF 2016, ARI 2020, NYG 2016 at 0.92-0.93).

**3. A new head coach moves a team's style; a new QB alone does not detectably.** Season-to-season style shift above the "no change" baseline (3.3 robust-z units): new coach only **+0.66** [+0.18, +1.17], new coach and new QB **+0.95** [+0.28, +1.68], new QB only +0.15 [-0.12, +0.43]. The largest single-season shifts are Washington 2024, Arizona 2023 (Kingsbury to Gannon), San Francisco 2017 (Kelly to Shanahan), Arizona 2019 (Kingsbury arrives) and the Giants 2018 (McAdoo to Shurmur).

**4. Offenses differ in execution, not in the style choices we can measure.** Splitting each team-season's points per drive vs. the league: **execution explains ~93%** of the variance, situation (field position, score, opponent) ~9%, and **opening-style choices essentially none** (spread of 0.02 points/drive vs. 0.49 for execution). Style matters on a drive (a deep-shot opener is worth about +0.36), but teams barely differ in how often they use it, so it contributes almost nothing to the gap between offenses. The best execution seasons include MVP-caliber QB years (KC 2018, ATL 2016, GB 2020, NE 2017); the worst are CLE 2024, ARI 2018, TEN 2025 and NYJ 2023.

**Caveats**
- **Coordinators are invisible.** Head-coach and primary-QB changes are the only events we can see, and coach changes bring staff changes with them. "Coach effect" here means "change of regime," not the head coach personally.
- **Noise.** A team-season is ~143 drives. One season's execution estimate has a standard error of ~0.24 points/drive against a spread of 0.49, and the "no change" baseline for style shift is itself large (3.3), so individual-season rankings and shifts are suggestive, not precise.
- **Execution is a residual.** It contains QB, talent, scheme, injuries and luck, and the style component is Phase 4's *association*, not a proven causal effect. In-sample fits.
